In [1]:
import os
import sys
import time
import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# Path setup (assume this notebook runs under tagging/pure_unsmear)
THIS_DIR = os.getcwd()
MODULE_DIR = THIS_DIR
TAGGING_DIR = os.path.abspath(os.path.join(MODULE_DIR, '..'))

sys.path.insert(0, MODULE_DIR)

import tool  # noqa: E402
from model import TokenMLP, TokenTransformerRegressor, TokenUNet1D, CondFlowMatcher  # noqa: E402

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

RUN_NAME = 'pure_unsmear_axisown_mlp_transformer_unet_fm'
OUT_DIR = os.path.join(MODULE_DIR, 'runs', RUN_NAME)
FIG_DIR = os.path.join(OUT_DIR, 'figs')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
SHARED_BASELINE_DIR = os.path.join(MODULE_DIR, 'runs', 'shared_offline_hlt_baselines')
SHARED_BASELINE_CKPT_DIR = os.path.join(SHARED_BASELINE_DIR, 'ckpts')

tool.ensure_dir(FIG_DIR)
tool.ensure_dir(CKPT_DIR)
tool.ensure_dir(SHARED_BASELINE_CKPT_DIR)

CONFIG = {
    'data_path': os.path.join(TAGGING_DIR, 'test.h5'),
    'load_shared_baselines': True,
    'load_unsmear_bundle': False,
    'n_jets': 200000,
    'max_particles': 100,
    # Unsmear training features: '3d'/'4d'/'7d'
    'feature_kind': '7d',
    # Use 7d uniformly for downstream tagger AUC comparisons
    'downstream_feature_kind': '7d',
    # Source of the last 3 dimensions in downstream 7d: 'direct' or 'rebuild_from_first4'
    'downstream_post3_source': 'direct',
    'consistency': {
        'w_dr': 0.0,
    },
    'uncertainty': {
        'enabled': False,
        'on_features': ['log_E', 'log_pt_rel', 'log_E_rel', 'dR'],
        'log_var_clip': 6.0,
    },
    'hlt_effects': {
        # Pure unsmear: use the same threshold=0.5 for offline / HLT and keep only smearing effects
        'pt_threshold_offline': 0.5,
        'pt_threshold_hlt': 0.5,
        'pt_resolution': 0.10,
        'eta_resolution': 0.03,
        'phi_resolution': 0.03,
    },
    'mlp': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'hidden_dim': 256,
        'num_layers': 4,
        'dropout': 0.1,
        'return_reco': True,
        'predict_logvar': False,
        # Mask option: if False, it degenerates to using the mask only in the loss
        'add_mask_channel': False,
        'mask_output': True,
    },
    'transformer': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 4,
        'ff_dim': 512,
        'dropout': 0.1,
        'return_reco': True,
        'predict_logvar': False,
        'add_mask_channel': True,
        'mask_output': True,
        'use_positional_embedding': True,
        'max_seq_len': 128,
    },
    'unet': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'base_channels': 64,
        'depth': 3,
        'dropout': 0.1,
        'return_reco': True,
        'predict_logvar': False,
        'add_mask_channel': False,
        'mask_output': True,
        # U-Net-specific optimization: repeatedly apply the mask on encoder / decoder feature maps
        'mask_encoder_features': True,
    },
    'fm': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 4,
        'ff_dim': 512,
        'dropout': 0.1,
        'time_n_freqs': 16,
        'time_max_freq': 200.0,
        'time_t_eps': 1e-4,
        'per_layer_cond': True,
        'sampler': 'heun',
        'sample_steps': 30,
    },
    'training': {
        'batch_size': 256,
        'epochs': 40,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 3,
        'patience': 6,
        'resmear_each_epoch': True,
        'resmear_seed_stride': 1,
    },
}

# Automatically align feature dimensions
feat_names = tool.get_feat_names(CONFIG['feature_kind'])
D = len(feat_names)
CONFIG['mlp']['input_dim'] = D
CONFIG['transformer']['input_dim'] = D
CONFIG['transformer']['max_seq_len'] = int(CONFIG['max_particles'])
CONFIG['unet']['input_dim'] = D
CONFIG['fm']['input_dim'] = D
CONFIG['mlp']['predict_logvar'] = bool(CONFIG['uncertainty']['enabled'])
CONFIG['transformer']['predict_logvar'] = bool(CONFIG['uncertainty']['enabled'])
CONFIG['unet']['predict_logvar'] = bool(CONFIG['uncertainty']['enabled'])

print('Data path:', CONFIG['data_path'])
print('Run dir:', OUT_DIR)
print('Feature kind:', CONFIG['feature_kind'], 'input_dim:', D, 'feat_names:', feat_names)

Device: cuda
Data path: d:\PracticeTagging\tagging\test.h5
Run dir: d:\PracticeTagging\tagging\pure_unsmear\runs\pure_unsmear_axisown_mlp_transformer_unet_fm
Feature kind: 7d input_dim: 7 feat_names: ['dEta', 'dPhi', 'log_pt', 'log_E', 'log_pt_rel', 'log_E_rel', 'dR']


In [2]:
# Load raw constituents
n = int(CONFIG['n_jets'])
S = int(CONFIG['max_particles'])

with h5py.File(CONFIG['data_path'], 'r') as f:
    labels = f['labels'][:n].astype(np.int64)
    weights = f['weights'][:n].astype(np.float32)
    pt = f['fjet_clus_pt'][:n, :S].astype(np.float32)
    eta = f['fjet_clus_eta'][:n, :S].astype(np.float32)
    phi = f['fjet_clus_phi'][:n, :S].astype(np.float32)
    E = f['fjet_clus_E'][:n, :S].astype(np.float32)

constituents_raw = np.stack([pt, eta, phi, E], axis=-1)  # [N,S,4]
mask_raw = pt > 0

print('Raw:', constituents_raw.shape, 'mask:', mask_raw.shape)
print('Signal:', int(labels.sum()), 'Bkg:', int((1 - labels).sum()))


Raw: (200000, 100, 4) mask: (200000, 100)
Signal: 99836 Bkg: 100164


In [3]:
# Build supervision pairs: target=unsmeared view, input=smeared HLT view
hcfg = tool.HLTEffectsCfg(**CONFIG['hlt_effects'])

pre_const, post_const, post_mask = tool.apply_hlt_effects_pair(
    constituents_raw,
    mask_raw,
    hcfg,
    seed=seed,
) 
# Note: HLT E depends on pt and eta.

pre_const = pre_const.copy()
post_const = post_const.copy()
pre_const[~post_mask] = 0.0
post_const[~post_mask] = 0.0
assert np.all(pre_const[~post_mask] == 0.0) and np.all(post_const[~post_mask] == 0.0)

print('Pre/post:', pre_const.shape, post_const.shape, 'mask:', post_mask.shape)
print('Avg tokens per jet (post):', float(post_mask.sum(axis=1).mean()))

axis_pre = tool.compute_jet_axis(pre_const, post_mask)
axis_post = tool.compute_jet_axis(post_const, post_mask)
feat_pre = tool.compute_features_with_axis(pre_const, post_mask, axis_pre, kind=CONFIG['feature_kind'])  # Use post_mask here because the current thresholds are identical, so the same post_mask can be shared.
feat_post = tool.compute_features_with_axis(post_const, post_mask, axis_post, kind=CONFIG['feature_kind'])

print('Features:', feat_pre.shape, feat_post.shape, 'kind:', CONFIG['feature_kind'])

# split (jet-level)
idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=seed, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
print(f"Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")


Pre/post: (200000, 100, 4) (200000, 100, 4) mask: (200000, 100)
Avg tokens per jet (post): 54.29243
Features: (200000, 100, 7) (200000, 100, 7) kind: 7d
Split: train=140,000 val=30,000 test=30,000


In [4]:
# Standardize using unsmeared-target train statistics
feat_means, feat_stds = tool.get_stats(feat_pre, post_mask, train_idx)

x_train = tool.standardize(feat_post, post_mask, feat_means, feat_stds, clip=10.0)
y_train = tool.standardize(feat_pre, post_mask, feat_means, feat_stds, clip=10.0)

print('Standardization done.')
print('Means:', np.round(feat_means, 4))
print('Stds :', np.round(feat_stds, 4))


Standardization done.
Means: [-2.0000e-04 -1.0000e-04  8.7940e+00  9.0840e+00 -5.2585e+00 -5.2701e+00
  2.2250e-01]
Stds : [0.2121 0.2173 1.5182 1.5217 1.4919 1.4935 0.2067]


In [5]:
# DataLoaders
BS = int(CONFIG['training']['batch_size'])

train_ds = tool.UnsmearJetDataset(x_train[train_idx], y_train[train_idx], post_mask[train_idx])
val_ds = tool.UnsmearJetDataset(x_train[val_idx], y_train[val_idx], post_mask[val_idx])
test_ds = tool.UnsmearJetDataset(x_train[test_idx], y_train[test_idx], post_mask[test_idx])

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BS)
test_loader = DataLoader(test_ds, batch_size=BS)

# During training, re-smear the inputs every epoch and recompute the post-side jet axis / features.
train_const_raw = constituents_raw[train_idx]
train_mask_raw = mask_raw[train_idx]


def make_epoch_train_loader(epoch: int):
    if not bool(CONFIG['training'].get('resmear_each_epoch', False)):
        return train_loader
    stride = int(CONFIG['training'].get('resmear_seed_stride', 1))
    epoch_seed = int(seed + (int(epoch) - 1) * stride)
    x_ep, y_ep, m_ep = tool.build_unsmear_epoch_arrays(
        train_const_raw,
        train_mask_raw,
        hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=epoch_seed,
        clip=10.0,
    )
    ds = tool.UnsmearJetDataset(x_ep, y_ep, m_ep)
    return DataLoader(ds, batch_size=BS, shuffle=True, drop_last=True)


print('Loaders ready, BS=', BS)
print('Epoch resmear enabled:', bool(CONFIG['training'].get('resmear_each_epoch', False)))

Loaders ready, BS= 256
Epoch resmear enabled: True


In [6]:
# Train MLP / Transformer / UNet regression vs Flow Matching
train_cfg = CONFIG['training']
epochs = int(train_cfg['epochs'])
lr = float(train_cfg['lr'])
wd = float(train_cfg['weight_decay'])
warm = int(train_cfg['warmup_epochs'])
pat = int(train_cfg['patience'])


def make_scheduler(opt):
    def lr_lambda(ep):
        if ep < warm:
            return float(ep + 1) / float(max(1, warm))
        t = (ep - warm) / float(max(1, epochs - warm))
        return 0.5 * (1.0 + np.cos(np.pi * t))

    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


# loss helpers
use_unc = bool(CONFIG['uncertainty']['enabled'])
w_dr = float(CONFIG['consistency']['w_dr'])

# feature indices
idx_map = {n: i for i, n in enumerate(feat_names)}
dphi_idx = idx_map.get('dPhi', None)
deta_idx = idx_map.get('dEta', None)
dr_idx = idx_map.get('dR', None)

# The dPhi wrap-aware residual must map the standardized space back to angle space.
if dphi_idx is not None:
    dphi_mean = float(feat_means[int(dphi_idx)])
    dphi_scale = float(feat_stds[int(dphi_idx)])
else:
    dphi_mean = 0.0
    dphi_scale = 1.0

# Enable uncertainty only on selected dimensions; the others are equivalent to log_var=0.
active_dim_mask = None
if use_unc:
    on = set(CONFIG['uncertainty'].get('on_features', []))
    active_dim_mask = torch.tensor([name in on for name in feat_names], dtype=torch.bool)


def regression_loss(mu: torch.Tensor, y: torch.Tensor, m: torch.Tensor, *, log_var: torch.Tensor | None = None) -> torch.Tensor:
    # Angle term: use a wrap-aware residual for dPhi.
    if use_unc:
        assert log_var is not None
        clip = float(CONFIG['uncertainty'].get('log_var_clip', 6.0))
        if dphi_idx is not None:
            base = tool.masked_gaussian_nll_wrap_dphi(
                mu,
                log_var,
                y,
                m,
                dphi_idx=int(dphi_idx),
                dphi_scale=dphi_scale,
                active_dim_mask=active_dim_mask,
                log_var_clip=clip,
            )
        else:
            base = tool.masked_gaussian_nll(
                mu,
                log_var,
                y,
                m,
                active_dim_mask=active_dim_mask,
                log_var_clip=clip,
            )
    else:
        if dphi_idx is not None:
            base = tool.masked_smooth_l1_wrap_dphi(mu, y, m, dphi_idx=int(dphi_idx), dphi_scale=dphi_scale)
        else:
            base = tool.masked_smooth_l1(mu, y, m)

    # Consistency term: enable it only for 7d features when dR is present.
    if (w_dr > 0.0) and (dr_idx is not None) and (deta_idx is not None) and (dphi_idx is not None):
        dR_cons = torch.sqrt(mu[..., int(deta_idx)] ** 2 + mu[..., int(dphi_idx)] ** 2 + 1e-12)
        dR_pred = mu[..., int(dr_idx)]
        cons = tool.masked_smooth_l1(dR_pred.unsqueeze(-1), dR_cons.unsqueeze(-1), m)
        base = base + float(w_dr) * cons

    return base


@torch.no_grad()
def eval_regression(model, loader):
    model.eval()
    tot = 0.0
    n = 0
    for batch in loader:
        x = batch['x'].to(device)
        y = batch['y'].to(device)
        m = batch['mask'].to(device)
        out = model(x, m)
        if isinstance(out, tuple):
            mu, log_var = out
        else:
            mu, log_var = out, None
        loss = regression_loss(mu, y, m, log_var=log_var)
        tot += float(loss.item()) * int(x.shape[0])
        n += int(x.shape[0])
    return tot / max(1, n)


@torch.no_grad()
def eval_fm(model, loader, *, steps: int):
    model.eval()
    tot = 0.0
    n = 0
    for batch in loader:
        x0 = batch['x'].to(device)
        y = batch['y'].to(device)
        m = batch['mask'].to(device)
        sampler = str(CONFIG['fm'].get('sampler', 'euler')).lower()
        if sampler == 'heun':
            pred = tool.fm_sample_heun(model, x0=x0, cond=x0, mask=m, steps=int(steps), dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        else:
            pred = tool.fm_sample_euler(model, x0=x0, cond=x0, mask=m, steps=int(steps), dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        if dphi_idx is not None:
            loss = tool.masked_smooth_l1_wrap_dphi(pred, y, m, dphi_idx=int(dphi_idx), dphi_scale=dphi_scale)
        else:
            loss = tool.masked_smooth_l1(pred, y, m)
        tot += float(loss.item()) * int(x0.shape[0])
        n += int(x0.shape[0])
    return tot / max(1, n)


with torch.no_grad():
    x_hlt_test = torch.from_numpy(x_train[test_idx]).to(device)
    y_off_test = torch.from_numpy(y_train[test_idx]).to(device)
    m_test = torch.from_numpy(post_mask[test_idx]).to(device)
    if dphi_idx is not None:
        hlt_test_loss = tool.masked_smooth_l1_wrap_dphi(
            x_hlt_test,
            y_off_test,
            m_test,
            dphi_idx=int(dphi_idx),
            dphi_scale=dphi_scale,
        )
    else:
        hlt_test_loss = tool.masked_smooth_l1(x_hlt_test, y_off_test, m_test)
    print('HLT->OFF test baseline loss (masked_smooth_l1_wrap_dphi):', float(hlt_test_loss.item()))


def train_regression_model(name: str, model, ckpt_name: str):
    ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
    if bool(CONFIG.get('load_unsmear_bundle', False)) and os.path.isfile(ckpt_path):
        obj = torch.load(ckpt_path, map_location='cpu')
        state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
        model.load_state_dict(state)
        print('Loaded checkpoint:', ckpt_path)
        return model

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = make_scheduler(opt)

    best_val, best_state, no_imp = 1e9, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        n = 0
        epoch_train_loader = make_epoch_train_loader(ep)
        for batch in epoch_train_loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)
            m = batch['mask'].to(device)

            opt.zero_grad(set_to_none=True)
            out = model(x, m)
            if isinstance(out, tuple):
                mu, log_var = out
            else:
                mu, log_var = out, None
            loss = regression_loss(mu, y, m, log_var=log_var)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tot += float(loss.item()) * int(x.shape[0])
            n += int(x.shape[0])

        train_loss = tot / max(1, n)
        val_loss = eval_regression(model, val_loader)
        sch.step()

        if val_loss < best_val - 1e-5:
            best_val = float(val_loss)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if ep == 1 or ep % 5 == 0:
            dt = time.time() - t0
            print(f"[{name}] ep={ep:03d} train={train_loss:.6f} val={val_loss:.6f} best={best_val:.6f} no_imp={no_imp} time={dt:.1f}s")
        if no_imp >= pat:
            print(f'[{name}] Early stopping')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save({'state_dict': model.state_dict(), 'best_val': best_val, 'config': CONFIG}, ckpt_path)
    print('Saved checkpoint:', ckpt_path)

    test_loss = eval_regression(model, test_loader)
    print(f'{name} test loss:', float(test_loss))
    return model


def train_fm():
    fm_cfg = dict(CONFIG['fm'])
    steps = int(fm_cfg.pop('sample_steps'))
    _ = fm_cfg.pop('sampler', None)
    model = CondFlowMatcher(**fm_cfg).to(device)
    ckpt_path = os.path.join(CKPT_DIR, 'unsmear_fm.pt')
    if bool(CONFIG.get('load_unsmear_bundle', False)) and os.path.isfile(ckpt_path):
        obj = torch.load(ckpt_path, map_location='cpu')
        state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
        model.load_state_dict(state)
        print('Loaded checkpoint:', ckpt_path)
        return model

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = make_scheduler(opt)

    best_val, best_state, no_imp = 1e9, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        n = 0
        epoch_train_loader = make_epoch_train_loader(ep)
        for batch in epoch_train_loader:
            x_post = batch['x'].to(device)
            x_pre = batch['y'].to(device)
            m = batch['mask'].to(device)

            t = torch.rand((x_post.size(0),), device=device, dtype=x_post.dtype)
            x_t, v = tool.fm_make_bridge(
                x_post,
                x_pre,
                t,
                dphi_idx=dphi_idx,
                dphi_mean=dphi_mean,
                dphi_scale=dphi_scale,
            )

            opt.zero_grad(set_to_none=True)
            v_hat = model(x_t, x_post, m, t)
            loss = tool.masked_mse(v_hat, v, m)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tot += float(loss.item()) * int(x_post.shape[0])
            n += int(x_post.shape[0])

        train_loss = tot / max(1, n)
        val_loss = eval_fm(model, val_loader, steps=steps)
        sch.step()

        if val_loss < best_val - 1e-5:
            best_val = float(val_loss)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if ep == 1 or ep % 5 == 0:
            dt = time.time() - t0
            print(f"[FM] ep={ep:03d} train={train_loss:.6f} val={val_loss:.6f} best={best_val:.6f} no_imp={no_imp} time={dt:.1f}s")
        if no_imp >= pat:
            print('[FM] Early stopping')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save({'state_dict': model.state_dict(), 'best_val': best_val, 'config': CONFIG}, ckpt_path)
    print('Saved checkpoint:', ckpt_path)

    test_loss = eval_fm(model, test_loader, steps=steps)
    print('FM test loss:', float(test_loss))
    return model


mlp = train_regression_model('MLP', TokenMLP(**CONFIG['mlp']).to(device), 'unsmear_mlp.pt')
transformer = train_regression_model('Transformer', TokenTransformerRegressor(**CONFIG['transformer']).to(device), 'unsmear_transformer.pt')
unet = train_regression_model('UNet', TokenUNet1D(**CONFIG['unet']).to(device), 'unsmear_unet.pt')
fm = train_fm()


HLT->OFF test baseline loss (masked_smooth_l1_wrap_dphi): 0.04267311096191406


KeyboardInterrupt: 

In [ ]:
# Visualizations
# Use the `feat_names` automatically generated in cell 0 from `feature_kind`.

@torch.no_grad()
def collect_all(loader):
    """Collect x/y/mask from a loader."""
    ys = []
    xs = []
    masks = []
    for batch in loader:
        x = batch['x'].to(device)
        m = batch['mask'].to(device)
        y = batch['y'].to(device)
        ys.append(y.cpu().numpy())
        xs.append(x.cpu().numpy())
        masks.append(m.cpu().numpy())

    return {
        'y': np.concatenate(ys, axis=0),
        'x': np.concatenate(xs, axis=0),
        'mask': np.concatenate(masks, axis=0),
    }


@torch.no_grad()
def predict_regression(model, loader):
    model.eval()
    outs = []
    for batch in loader:
        x = batch['x'].to(device)
        m = batch['mask'].to(device)
        out = model(x, m)
        mu = out[0] if isinstance(out, tuple) else out
        outs.append(mu.cpu().numpy())
    return np.concatenate(outs, axis=0)


@torch.no_grad()
def predict_fm(model, loader):
    model.eval()
    outs = []
    fm_steps = int(CONFIG['fm']['sample_steps'])
    fm_sampler = str(CONFIG['fm'].get('sampler', 'euler')).lower()
    for batch in loader:
        x0 = batch['x'].to(device)
        m = batch['mask'].to(device)
        if fm_sampler == 'heun':
            pred = tool.fm_sample_heun(model, x0=x0, cond=x0, mask=m, steps=fm_steps, dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        else:
            pred = tool.fm_sample_euler(model, x0=x0, cond=x0, mask=m, steps=fm_steps, dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        outs.append(pred.cpu().numpy())
    return np.concatenate(outs, axis=0)


pack = collect_all(test_loader)
y_std = pack['y']
x_std = pack['x']
mask_np = pack['mask']

pred_mlp = predict_regression(mlp, test_loader)
pred_transformer = predict_regression(transformer, test_loader)
pred_unet = predict_regression(unet, test_loader)
pred_fm = predict_fm(fm, test_loader)

# 1) per-feature residual distributions (before vs MLP vs Transformer vs UNet vs FM)
for i, name in enumerate(feat_names):
    x_i = x_std[..., i][mask_np]
    y_i = y_std[..., i][mask_np]
    m_i = pred_mlp[..., i][mask_np]
    t_i = pred_transformer[..., i][mask_np]
    u_i = pred_unet[..., i][mask_np]
    f_i = pred_fm[..., i][mask_np]

    r_before = x_i - y_i
    r_mlp = m_i - y_i
    r_transformer = t_i - y_i
    r_unet = u_i - y_i
    r_fm = f_i - y_i

    if name == 'dPhi':
        sc = float(feat_stds[i])
        r_before = tool.wrap_dphi_np(r_before * sc) / sc
        r_mlp = tool.wrap_dphi_np(r_mlp * sc) / sc
        r_transformer = tool.wrap_dphi_np(r_transformer * sc) / sc
        r_unet = tool.wrap_dphi_np(r_unet * sc) / sc
        r_fm = tool.wrap_dphi_np(r_fm * sc) / sc

    plt.figure(figsize=(6.8, 4.6))
    bins = 120
    plt.hist(r_before, bins=bins, density=True, alpha=0.30, label='Before (post - pre)')
    plt.hist(r_mlp, bins=bins, density=True, alpha=0.30, label='MLP (pred - pre)')
    plt.hist(r_transformer, bins=bins, density=True, alpha=0.30, label='Transformer (pred - pre)')
    plt.hist(r_unet, bins=bins, density=True, alpha=0.30, label='UNet (pred - pre)')
    plt.hist(r_fm, bins=bins, density=True, alpha=0.30, label='FM (pred - pre)')
    plt.title(f'Residual compare: {name}')
    plt.xlabel('Residual (std space)')
    plt.ylabel('Density')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f'resid_compare_{name}.png')
    plt.savefig(fig_path, dpi=160, bbox_inches='tight')
    print('Saved figure:', fig_path)
    plt.show()

# 2) pred vs true scatter (MLP / Transformer / UNet / FM)
N_scatter = 100000
sel = np.where(mask_np.reshape(-1))[0]
if sel.size > N_scatter:
    sel = np.random.RandomState(0).choice(sel, size=N_scatter, replace=False)

for i, name in enumerate(feat_names):
    y_flat = y_std[..., i].reshape(-1)[sel]
    m_flat_pred = pred_mlp[..., i].reshape(-1)[sel]
    t_flat = pred_transformer[..., i].reshape(-1)[sel]
    u_flat = pred_unet[..., i].reshape(-1)[sel]
    f_flat = pred_fm[..., i].reshape(-1)[sel]

    lo = np.percentile(np.concatenate([y_flat, m_flat_pred, t_flat, u_flat, f_flat]), 1)
    hi = np.percentile(np.concatenate([y_flat, m_flat_pred, t_flat, u_flat, f_flat]), 99)

    plt.figure(figsize=(21.0, 4.8))
    plt.subplot(1, 4, 1)
    plt.hexbin(y_flat, m_flat_pred, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('MLP pred (std)')
    plt.title(f'MLP: {name}')
    plt.grid(True, alpha=0.2)

    plt.subplot(1, 4, 2)
    plt.hexbin(y_flat, t_flat, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('Transformer pred (std)')
    plt.title(f'Transformer: {name}')
    plt.grid(True, alpha=0.2)

    plt.subplot(1, 4, 3)
    plt.hexbin(y_flat, u_flat, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('UNet pred (std)')
    plt.title(f'UNet: {name}')
    plt.grid(True, alpha=0.2)

    plt.subplot(1, 4, 4)
    plt.hexbin(y_flat, f_flat, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('FM pred (std)')
    plt.title(f'FM: {name}')
    plt.grid(True, alpha=0.2)

    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f'scatter_compare_{name}.png')
    plt.savefig(fig_path, dpi=160, bbox_inches='tight')
    print('Saved figure:', fig_path)
    plt.show()

# 3) Quantitative tables (MAE/RMSE + tail quantiles) and binned bias curves
res_before = x_std - y_std
res_mlp = pred_mlp - y_std
res_transformer = pred_transformer - y_std
res_unet = pred_unet - y_std
res_fm = pred_fm - y_std

m_flat = mask_np.reshape(-1)
splits = [('all', m_flat)]


def _metrics(res_1d: np.ndarray):
    if res_1d.size == 0:
        return {'bias': np.nan, 'mae': np.nan, 'rmse': np.nan, 'p50': np.nan, 'p90': np.nan, 'p99': np.nan}
    abs_r = np.abs(res_1d)
    return {
        'bias': float(np.mean(res_1d)),
        'mae': float(np.mean(abs_r)),
        'rmse': float(np.sqrt(np.mean(res_1d ** 2))),
        'p50': float(np.quantile(abs_r, 0.50)),
        'p90': float(np.quantile(abs_r, 0.90)),
        'p99': float(np.quantile(abs_r, 0.99)),
    }


def print_metrics_table(split_name: str, sel_flat: np.ndarray):
    print('\n' + '=' * 80)
    print(f"Metrics summary (std space) | split={split_name} | n_tokens={int(sel_flat.sum())}")
    print('=' * 80)
    header = ['feature', 'method', 'bias', 'mae', 'rmse', 'abs_p50', 'abs_p90', 'abs_p99']
    print('\t'.join(header))

    for i, name in enumerate(feat_names):
        y_flat = y_std[..., i].reshape(-1)[sel_flat]
        if y_flat.size == 0:
            continue
        for method, res in [('before', res_before), ('mlp', res_mlp), ('transformer', res_transformer), ('unet', res_unet), ('fm', res_fm)]:
            r = res[..., i].reshape(-1)[sel_flat]
            if name == 'dPhi':
                sc = float(feat_stds[i])
                r = tool.wrap_dphi_np(r * sc) / sc
            mm = _metrics(r)
            print('\t'.join([
                name,
                method,
                f"{mm['bias']:.6f}",
                f"{mm['mae']:.6f}",
                f"{mm['rmse']:.6f}",
                f"{mm['p50']:.6f}",
                f"{mm['p90']:.6f}",
                f"{mm['p99']:.6f}",
            ]))


for split_name, sel_flat in splits:
    print_metrics_table(split_name, sel_flat)


def plot_binned_bias(feature_idx: int, feature_name: str, *, sel_flat: np.ndarray, split_name: str, n_bins: int = 20):
    y_flat = y_std[..., feature_idx].reshape(-1)[sel_flat]
    if y_flat.size < 1000:
        return

    qs = np.linspace(0.0, 1.0, int(n_bins) + 1)
    edges = np.quantile(y_flat, qs)
    edges = np.unique(edges)
    if edges.size < 3:
        return

    centers = 0.5 * (edges[:-1] + edges[1:])

    def _binned_stats(res_arr: np.ndarray):
        r_flat = res_arr[..., feature_idx].reshape(-1)[sel_flat]
        if feature_name == 'dPhi':
            sc = float(feat_stds[int(feature_idx)])
            r_flat = tool.wrap_dphi_np(r_flat * sc) / sc
        means = []
        stds = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            inb = (y_flat >= lo) & (y_flat < hi)
            if not np.any(inb):
                means.append(np.nan)
                stds.append(np.nan)
            else:
                means.append(float(np.mean(r_flat[inb])))
                stds.append(float(np.std(r_flat[inb])))
        return np.asarray(means), np.asarray(stds)

    mb, sb = _binned_stats(res_before)
    mm, sm = _binned_stats(res_mlp)
    mt, st = _binned_stats(res_transformer)
    mu, su = _binned_stats(res_unet)
    mf, sf = _binned_stats(res_fm)

    plt.figure(figsize=(7.2, 4.8))
    plt.plot(centers, mb, label='Before', lw=2)
    plt.fill_between(centers, mb - sb, mb + sb, alpha=0.12)

    plt.plot(centers, mm, label='MLP', lw=2)
    plt.fill_between(centers, mm - sm, mm + sm, alpha=0.12)

    plt.plot(centers, mt, label='Transformer', lw=2)
    plt.fill_between(centers, mt - st, mt + st, alpha=0.12)

    plt.plot(centers, mu, label='UNet', lw=2)
    plt.fill_between(centers, mu - su, mu + su, alpha=0.12)

    plt.plot(centers, mf, label='FM', lw=2)
    plt.fill_between(centers, mf - sf, mf + sf, alpha=0.12)

    plt.axhline(0.0, color='k', lw=1, alpha=0.6)
    plt.xlabel(f'True {feature_name} (std)')
    plt.ylabel('Mean residual (pred - true)')
    plt.title(f'Binned bias: {feature_name} | {split_name}')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    out = os.path.join(FIG_DIR, f'bias_{feature_name}_{split_name}.png')
    plt.savefig(out, dpi=160, bbox_inches='tight')
    print('Saved figure:', out)
    plt.show()


for split_name, sel_flat in splits:
    for i, name in enumerate(feat_names):
        plot_binned_bias(i, name, sel_flat=sel_flat, split_name=split_name, n_bins=20)


In [ ]:
# Downstream tagger evaluation
# In the current pure setup, the offline view and the unsmeared target are equivalent in their own axis frames,
# so we keep the offline view as the common reference input here.
# Downstream comparisons include: OFF teacher / HLT / MLP-unsmear / UNet-unsmear / FM-unsmear and their KD variants.

import hashlib
from model import ParticleTransformerKD  # noqa: E402

baseline_tool = tool

# -----------------------------
# 1) Build unsmeared views for train/val/test
# -----------------------------
@torch.no_grad()
def _predict_reg_model(model, x_std_np: np.ndarray, mask_np: np.ndarray, *, bs: int = 512) -> np.ndarray:
    model.eval()
    ds = tool.UnsmearJetDataset(x_std_np, x_std_np, mask_np)
    ld = DataLoader(ds, batch_size=int(bs), shuffle=False)
    outs = []
    for batch in ld:
        x = batch['x'].to(device)
        m = batch['mask'].to(device)
        out = model(x, m)
        mu = out[0] if isinstance(out, tuple) else out
        outs.append(mu.cpu().numpy())
    return np.concatenate(outs, axis=0)


@torch.no_grad()
def _predict_fm_model(model, x_std_np: np.ndarray, mask_np: np.ndarray, *, bs: int = 512) -> np.ndarray:
    model.eval()
    ds = tool.UnsmearJetDataset(x_std_np, x_std_np, mask_np)
    ld = DataLoader(ds, batch_size=int(bs), shuffle=False)
    outs = []
    steps = int(CONFIG['fm']['sample_steps'])
    sampler = str(CONFIG['fm'].get('sampler', 'euler')).lower()
    for batch in ld:
        x0 = batch['x'].to(device)
        m = batch['mask'].to(device)
        if sampler == 'heun':
            pred = tool.fm_sample_heun(model, x0=x0, cond=x0, mask=m, steps=steps, dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        else:
            pred = tool.fm_sample_euler(model, x0=x0, cond=x0, mask=m, steps=steps, dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        outs.append(pred.cpu().numpy())
    return np.concatenate(outs, axis=0)


print('Building MLP-unsmeared features...')
mlp_train = _predict_reg_model(mlp, x_train[train_idx], post_mask[train_idx], bs=512)
mlp_val = _predict_reg_model(mlp, x_train[val_idx], post_mask[val_idx], bs=512)
mlp_test = _predict_reg_model(mlp, x_train[test_idx], post_mask[test_idx], bs=512)
print('MLP-unsmeared feats:', mlp_train.shape, mlp_val.shape, mlp_test.shape)

print('Building Transformer-unsmeared features...')
transformer_train = _predict_reg_model(transformer, x_train[train_idx], post_mask[train_idx], bs=512)
transformer_val = _predict_reg_model(transformer, x_train[val_idx], post_mask[val_idx], bs=512)
transformer_test = _predict_reg_model(transformer, x_train[test_idx], post_mask[test_idx], bs=512)
print('Transformer-unsmeared feats:', transformer_train.shape, transformer_val.shape, transformer_test.shape)

print('Building UNet-unsmeared features...')
unet_train = _predict_reg_model(unet, x_train[train_idx], post_mask[train_idx], bs=512)
unet_val = _predict_reg_model(unet, x_train[val_idx], post_mask[val_idx], bs=512)
unet_test = _predict_reg_model(unet, x_train[test_idx], post_mask[test_idx], bs=512)
print('UNet-unsmeared feats:', unet_train.shape, unet_val.shape, unet_test.shape)

print('Building FM-unsmeared features...')
fm_train = _predict_fm_model(fm, x_train[train_idx], post_mask[train_idx], bs=512)
fm_val = _predict_fm_model(fm, x_train[val_idx], post_mask[val_idx], bs=512)
fm_test = _predict_fm_model(fm, x_train[test_idx], post_mask[test_idx], bs=512)
print('FM-unsmeared feats:', fm_train.shape, fm_val.shape, fm_test.shape)

# Offline reference: use its own axis frame; in the current pure setup this is equivalent to the unsmeared target.
pt_thr_off = float(CONFIG['hlt_effects'].get('pt_threshold_offline', 0.5))
off_mask_full = mask_raw & (constituents_raw[:, :, 0] >= pt_thr_off)
off_const_full = constituents_raw.copy()
off_const_full[~off_mask_full] = 0.0

DOWN_KIND = str(CONFIG.get('downstream_feature_kind', '7d')).lower()

axis_off = tool.compute_jet_axis(off_const_full, off_mask_full)
feat_off_full = tool.compute_features_with_axis(off_const_full, off_mask_full, axis_off, kind=DOWN_KIND)
off_means, off_stds = tool.get_stats(feat_off_full, off_mask_full, train_idx)
feat_off_full_std = tool.standardize(feat_off_full, off_mask_full, off_means, off_stds, clip=10.0)

# HLT uses its own axis frame.
feat_hlt_down = tool.compute_features_with_axis(post_const, post_mask, axis_post, kind=DOWN_KIND)
feat_hlt_std = tool.standardize(feat_hlt_down, post_mask, off_means, off_stds, clip=10.0)

# First restore each unsmear model output from the training standardized space back to raw feature space.

def _slice_axis(axis: dict, idx: np.ndarray) -> dict:
    return {k: np.asarray(v)[idx] for k, v in axis.items()}


def _restore_pred_raw(pred_std: np.ndarray) -> np.ndarray:
    raw = pred_std * feat_stds[None, None, :] + feat_means[None, None, :]
    raw = np.asarray(raw, dtype=np.float32)
    if raw.shape[-1] > 1:
        raw[..., 1] = tool.wrap_dphi_np(raw[..., 1])
    return raw


def _rebuild_post3_from_first4(raw_pred: np.ndarray, axis_split: dict, mask_split: np.ndarray) -> np.ndarray:
    rebuilt = raw_pred.copy().astype(np.float32)
    rebuilt7 = tool.feats_to_7d(raw_pred[..., :4], mask_split, axis_split, kind='4d')
    rebuilt[..., 4:7] = rebuilt7[..., 4:7]
    rebuilt[~mask_split] = 0.0
    return rebuilt.astype(np.float32)


def _to_downstream_std(raw_pred: np.ndarray, mask_split: np.ndarray) -> np.ndarray:
    return tool.standardize(raw_pred, mask_split, off_means, off_stds, clip=10.0)


def _print_post3_compare(label: str, direct_std: np.ndarray, rebuilt_std: np.ndarray, off_std: np.ndarray, mask_split: np.ndarray):
    if DOWN_KIND != '7d':
        return
    names7 = tool.get_feat_names('7d')
    idx7 = {n: i for i, n in enumerate(names7)}
    post3_names = ['log_pt_rel', 'log_E_rel', 'dR']
    print()
    print('=' * 90)
    print(f'{label} post-3D compare vs OFF (std space) | n_tokens={int(mask_split.sum())}')
    print('=' * 90)
    print('\t'.join(['feature', 'method', 'bias', 'mae', 'rmse', 'abs_p50', 'abs_p90', 'abs_p99']))
    for name in post3_names:
        i = int(idx7[name])
        y_i = off_std[..., i]
        direct_i = direct_std[..., i]
        rebuilt_i = rebuilt_std[..., i]
        for method, arr in [('direct', direct_i - y_i), ('rebuild_from_first4', rebuilt_i - y_i)]:
            mm = _metrics(arr[mask_split])
            print('\t'.join([
                name,
                method,
                f"{mm['bias']:.6f}",
                f"{mm['mae']:.6f}",
                f"{mm['rmse']:.6f}",
                f"{mm['p50']:.6f}",
                f"{mm['p90']:.6f}",
                f"{mm['p99']:.6f}",
            ]))


mlp_train_raw_direct = _restore_pred_raw(mlp_train)
mlp_val_raw_direct = _restore_pred_raw(mlp_val)
mlp_test_raw_direct = _restore_pred_raw(mlp_test)

transformer_train_raw_direct = _restore_pred_raw(transformer_train)
transformer_val_raw_direct = _restore_pred_raw(transformer_val)
transformer_test_raw_direct = _restore_pred_raw(transformer_test)

unet_train_raw_direct = _restore_pred_raw(unet_train)
unet_val_raw_direct = _restore_pred_raw(unet_val)
unet_test_raw_direct = _restore_pred_raw(unet_test)

fm_train_raw_direct = _restore_pred_raw(fm_train)
fm_val_raw_direct = _restore_pred_raw(fm_val)
fm_test_raw_direct = _restore_pred_raw(fm_test)

POST3_SOURCE = str(CONFIG.get('downstream_post3_source', 'direct')).lower()

if DOWN_KIND == '7d' and str(CONFIG['feature_kind']).lower() == '7d':
    axis_off_train = _slice_axis(axis_off, train_idx)
    axis_off_val = _slice_axis(axis_off, val_idx)
    axis_off_test = _slice_axis(axis_off, test_idx)

    mlp_train_raw_rebuilt = _rebuild_post3_from_first4(mlp_train_raw_direct, axis_off_train, post_mask[train_idx])
    mlp_val_raw_rebuilt = _rebuild_post3_from_first4(mlp_val_raw_direct, axis_off_val, post_mask[val_idx])
    mlp_test_raw_rebuilt = _rebuild_post3_from_first4(mlp_test_raw_direct, axis_off_test, post_mask[test_idx])

    transformer_train_raw_rebuilt = _rebuild_post3_from_first4(transformer_train_raw_direct, axis_off_train, post_mask[train_idx])
    transformer_val_raw_rebuilt = _rebuild_post3_from_first4(transformer_val_raw_direct, axis_off_val, post_mask[val_idx])
    transformer_test_raw_rebuilt = _rebuild_post3_from_first4(transformer_test_raw_direct, axis_off_test, post_mask[test_idx])

    unet_train_raw_rebuilt = _rebuild_post3_from_first4(unet_train_raw_direct, axis_off_train, post_mask[train_idx])
    unet_val_raw_rebuilt = _rebuild_post3_from_first4(unet_val_raw_direct, axis_off_val, post_mask[val_idx])
    unet_test_raw_rebuilt = _rebuild_post3_from_first4(unet_test_raw_direct, axis_off_test, post_mask[test_idx])

    fm_train_raw_rebuilt = _rebuild_post3_from_first4(fm_train_raw_direct, axis_off_train, post_mask[train_idx])
    fm_val_raw_rebuilt = _rebuild_post3_from_first4(fm_val_raw_direct, axis_off_val, post_mask[val_idx])
    fm_test_raw_rebuilt = _rebuild_post3_from_first4(fm_test_raw_direct, axis_off_test, post_mask[test_idx])

    mlp_test_std_direct = _to_downstream_std(mlp_test_raw_direct, post_mask[test_idx])
    mlp_test_std_rebuilt = _to_downstream_std(mlp_test_raw_rebuilt, post_mask[test_idx])
    transformer_test_std_direct = _to_downstream_std(transformer_test_raw_direct, post_mask[test_idx])
    transformer_test_std_rebuilt = _to_downstream_std(transformer_test_raw_rebuilt, post_mask[test_idx])
    unet_test_std_direct = _to_downstream_std(unet_test_raw_direct, post_mask[test_idx])
    unet_test_std_rebuilt = _to_downstream_std(unet_test_raw_rebuilt, post_mask[test_idx])
    fm_test_std_direct = _to_downstream_std(fm_test_raw_direct, post_mask[test_idx])
    fm_test_std_rebuilt = _to_downstream_std(fm_test_raw_rebuilt, post_mask[test_idx])

    _print_post3_compare('MLP', mlp_test_std_direct, mlp_test_std_rebuilt, feat_off_full_std[test_idx], post_mask[test_idx])
    _print_post3_compare('Transformer', transformer_test_std_direct, transformer_test_std_rebuilt, feat_off_full_std[test_idx], post_mask[test_idx])
    _print_post3_compare('UNET', unet_test_std_direct, unet_test_std_rebuilt, feat_off_full_std[test_idx], post_mask[test_idx])
    _print_post3_compare('FM', fm_test_std_direct, fm_test_std_rebuilt, feat_off_full_std[test_idx], post_mask[test_idx])

    use_rebuilt = POST3_SOURCE == 'rebuild_from_first4'
    mlp_train_raw = mlp_train_raw_rebuilt if use_rebuilt else mlp_train_raw_direct
    mlp_val_raw = mlp_val_raw_rebuilt if use_rebuilt else mlp_val_raw_direct
    mlp_test_raw = mlp_test_raw_rebuilt if use_rebuilt else mlp_test_raw_direct

    transformer_train_raw = transformer_train_raw_rebuilt if use_rebuilt else transformer_train_raw_direct
    transformer_val_raw = transformer_val_raw_rebuilt if use_rebuilt else transformer_val_raw_direct
    transformer_test_raw = transformer_test_raw_rebuilt if use_rebuilt else transformer_test_raw_direct

    unet_train_raw = unet_train_raw_rebuilt if use_rebuilt else unet_train_raw_direct
    unet_val_raw = unet_val_raw_rebuilt if use_rebuilt else unet_val_raw_direct
    unet_test_raw = unet_test_raw_rebuilt if use_rebuilt else unet_test_raw_direct

    fm_train_raw = fm_train_raw_rebuilt if use_rebuilt else fm_train_raw_direct
    fm_val_raw = fm_val_raw_rebuilt if use_rebuilt else fm_val_raw_direct
    fm_test_raw = fm_test_raw_rebuilt if use_rebuilt else fm_test_raw_direct
else:
    if DOWN_KIND == '7d' and str(CONFIG['feature_kind']).lower() != '7d':
        axis_off_train = _slice_axis(axis_off, train_idx)
        axis_off_val = _slice_axis(axis_off, val_idx)
        axis_off_test = _slice_axis(axis_off, test_idx)
        uns_kind = str(CONFIG['feature_kind']).lower()
        mlp_train_raw = tool.feats_to_7d(mlp_train_raw_direct, post_mask[train_idx], axis_off_train, kind=uns_kind)
        mlp_val_raw = tool.feats_to_7d(mlp_val_raw_direct, post_mask[val_idx], axis_off_val, kind=uns_kind)
        mlp_test_raw = tool.feats_to_7d(mlp_test_raw_direct, post_mask[test_idx], axis_off_test, kind=uns_kind)
        transformer_train_raw = tool.feats_to_7d(transformer_train_raw_direct, post_mask[train_idx], axis_off_train, kind=uns_kind)
        transformer_val_raw = tool.feats_to_7d(transformer_val_raw_direct, post_mask[val_idx], axis_off_val, kind=uns_kind)
        transformer_test_raw = tool.feats_to_7d(transformer_test_raw_direct, post_mask[test_idx], axis_off_test, kind=uns_kind)
        unet_train_raw = tool.feats_to_7d(unet_train_raw_direct, post_mask[train_idx], axis_off_train, kind=uns_kind)
        unet_val_raw = tool.feats_to_7d(unet_val_raw_direct, post_mask[val_idx], axis_off_val, kind=uns_kind)
        unet_test_raw = tool.feats_to_7d(unet_test_raw_direct, post_mask[test_idx], axis_off_test, kind=uns_kind)
        fm_train_raw = tool.feats_to_7d(fm_train_raw_direct, post_mask[train_idx], axis_off_train, kind=uns_kind)
        fm_val_raw = tool.feats_to_7d(fm_val_raw_direct, post_mask[val_idx], axis_off_val, kind=uns_kind)
        fm_test_raw = tool.feats_to_7d(fm_test_raw_direct, post_mask[test_idx], axis_off_test, kind=uns_kind)
    else:
        mlp_train_raw, mlp_val_raw, mlp_test_raw = mlp_train_raw_direct, mlp_val_raw_direct, mlp_test_raw_direct
        transformer_train_raw, transformer_val_raw, transformer_test_raw = transformer_train_raw_direct, transformer_val_raw_direct, transformer_test_raw_direct
        unet_train_raw, unet_val_raw, unet_test_raw = unet_train_raw_direct, unet_val_raw_direct, unet_test_raw_direct
        fm_train_raw, fm_val_raw, fm_test_raw = fm_train_raw_direct, fm_val_raw_direct, fm_test_raw_direct

mlp_train_std = tool.standardize(mlp_train_raw, post_mask[train_idx], off_means, off_stds, clip=10.0)
mlp_val_std = tool.standardize(mlp_val_raw, post_mask[val_idx], off_means, off_stds, clip=10.0)
mlp_test_std = tool.standardize(mlp_test_raw, post_mask[test_idx], off_means, off_stds, clip=10.0)

transformer_train_std = tool.standardize(transformer_train_raw, post_mask[train_idx], off_means, off_stds, clip=10.0)
transformer_val_std = tool.standardize(transformer_val_raw, post_mask[val_idx], off_means, off_stds, clip=10.0)
transformer_test_std = tool.standardize(transformer_test_raw, post_mask[test_idx], off_means, off_stds, clip=10.0)

unet_train_std = tool.standardize(unet_train_raw, post_mask[train_idx], off_means, off_stds, clip=10.0)
unet_val_std = tool.standardize(unet_val_raw, post_mask[val_idx], off_means, off_stds, clip=10.0)
unet_test_std = tool.standardize(unet_test_raw, post_mask[test_idx], off_means, off_stds, clip=10.0)

fm_train_std = tool.standardize(fm_train_raw, post_mask[train_idx], off_means, off_stds, clip=10.0)
fm_val_std = tool.standardize(fm_val_raw, post_mask[val_idx], off_means, off_stds, clip=10.0)
fm_test_std = tool.standardize(fm_test_raw, post_mask[test_idx], off_means, off_stds, clip=10.0)

print('Downstream feature kind:', DOWN_KIND, '| unsmear kind:', CONFIG['feature_kind'], '| post3 source:', POST3_SOURCE)

# Sanity check: see whether different unsmear outputs are closer to the offline space than HLT.
try:
    m_tr = post_mask[train_idx]
    mae_hlt = float(np.mean(np.abs((feat_hlt_std[train_idx] - feat_off_full_std[train_idx])[m_tr])))
    mae_mlp = float(np.mean(np.abs((mlp_train_std - feat_off_full_std[train_idx])[m_tr])))
    mae_transformer = float(np.mean(np.abs((transformer_train_std - feat_off_full_std[train_idx])[m_tr])))
    mae_unet = float(np.mean(np.abs((unet_train_std - feat_off_full_std[train_idx])[m_tr])))
    mae_fm = float(np.mean(np.abs((fm_train_std - feat_off_full_std[train_idx])[m_tr])))
    print('Sanity MAE vs OFF (train, downstream std): HLT=', mae_hlt, 'MLP=', mae_mlp, 'Transformer=', mae_transformer, 'UNET=', mae_unet, 'FM=', mae_fm)
except Exception as e:
    print('Sanity MAE vs OFF skipped:', repr(e))

# split tensors
y_tr, y_va, y_te = labels[train_idx], labels[val_idx], labels[test_idx]
w_tr, w_va, w_te = weights[train_idx], weights[val_idx], weights[test_idx]
m_hlt_train, m_hlt_val, m_hlt_test = post_mask[train_idx], post_mask[val_idx], post_mask[test_idx]
m_off_train, m_off_val, m_off_test = off_mask_full[train_idx], off_mask_full[val_idx], off_mask_full[test_idx]

train_ds_hlt = baseline_tool.JetDataset(feat_off_full_std[train_idx], feat_hlt_std[train_idx], y_tr, m_off_train, m_hlt_train, w_tr)
val_ds_hlt = baseline_tool.JetDataset(feat_off_full_std[val_idx], feat_hlt_std[val_idx], y_va, m_off_val, m_hlt_val, w_va)
test_ds_hlt = baseline_tool.JetDataset(feat_off_full_std[test_idx], feat_hlt_std[test_idx], y_te, m_off_test, m_hlt_test, w_te)

train_ds_mlp = baseline_tool.JetDataset(feat_off_full_std[train_idx], mlp_train_std, y_tr, m_off_train, m_hlt_train, w_tr)
val_ds_mlp = baseline_tool.JetDataset(feat_off_full_std[val_idx], mlp_val_std, y_va, m_off_val, m_hlt_val, w_va)
test_ds_mlp = baseline_tool.JetDataset(feat_off_full_std[test_idx], mlp_test_std, y_te, m_off_test, m_hlt_test, w_te)

train_ds_transformer = baseline_tool.JetDataset(feat_off_full_std[train_idx], transformer_train_std, y_tr, m_off_train, m_hlt_train, w_tr)
val_ds_transformer = baseline_tool.JetDataset(feat_off_full_std[val_idx], transformer_val_std, y_va, m_off_val, m_hlt_val, w_va)
test_ds_transformer = baseline_tool.JetDataset(feat_off_full_std[test_idx], transformer_test_std, y_te, m_off_test, m_hlt_test, w_te)

train_ds_unet = baseline_tool.JetDataset(feat_off_full_std[train_idx], unet_train_std, y_tr, m_off_train, m_hlt_train, w_tr)
val_ds_unet = baseline_tool.JetDataset(feat_off_full_std[val_idx], unet_val_std, y_va, m_off_val, m_hlt_val, w_va)
test_ds_unet = baseline_tool.JetDataset(feat_off_full_std[test_idx], unet_test_std, y_te, m_off_test, m_hlt_test, w_te)

train_ds_fm = baseline_tool.JetDataset(feat_off_full_std[train_idx], fm_train_std, y_tr, m_off_train, m_hlt_train, w_tr)
val_ds_fm = baseline_tool.JetDataset(feat_off_full_std[val_idx], fm_val_std, y_va, m_off_val, m_hlt_val, w_va)
test_ds_fm = baseline_tool.JetDataset(feat_off_full_std[test_idx], fm_test_std, y_te, m_off_test, m_hlt_test, w_te)

BS_TAG = 256
train_loader_hlt = DataLoader(train_ds_hlt, batch_size=BS_TAG, shuffle=True, drop_last=True)
val_loader_hlt = DataLoader(val_ds_hlt, batch_size=BS_TAG, shuffle=False)
test_loader_hlt = DataLoader(test_ds_hlt, batch_size=BS_TAG, shuffle=False)

train_loader_mlp = DataLoader(train_ds_mlp, batch_size=BS_TAG, shuffle=True, drop_last=True)
val_loader_mlp = DataLoader(val_ds_mlp, batch_size=BS_TAG, shuffle=False)
test_loader_mlp = DataLoader(test_ds_mlp, batch_size=BS_TAG, shuffle=False)

train_loader_transformer = DataLoader(train_ds_transformer, batch_size=BS_TAG, shuffle=True, drop_last=True)
val_loader_transformer = DataLoader(val_ds_transformer, batch_size=BS_TAG, shuffle=False)
test_loader_transformer = DataLoader(test_ds_transformer, batch_size=BS_TAG, shuffle=False)

train_loader_unet = DataLoader(train_ds_unet, batch_size=BS_TAG, shuffle=True, drop_last=True)
val_loader_unet = DataLoader(val_ds_unet, batch_size=BS_TAG, shuffle=False)
test_loader_unet = DataLoader(test_ds_unet, batch_size=BS_TAG, shuffle=False)

train_loader_fm = DataLoader(train_ds_fm, batch_size=BS_TAG, shuffle=True, drop_last=True)
val_loader_fm = DataLoader(val_ds_fm, batch_size=BS_TAG, shuffle=False)
test_loader_fm = DataLoader(test_ds_fm, batch_size=BS_TAG, shuffle=False)

# -----------------------------
# 2) Train/eval helpers (early stopping on val AUC)
# -----------------------------
TAGGER_CFG = {
    'input_dim': len(tool.get_feat_names(DOWN_KIND)),
    'embed_dim': 128,
    'num_heads': 8,
    'num_layers': 6,
    'ff_dim': 512,
    'dropout': 0.1,
}

TRAIN_TAG = {
    'epochs': 30,
    'lr': 5e-4,
    'weight_decay': 1e-5,
    'warmup_epochs': 3,
    'patience': 6,
    'grad_clip': 1.0,
}

KD_CFG = {
    'kd': {
        'temperature': 3.0,
        'alpha_kd': 0.5,
        'alpha_attn': 0.2,
    }
}


def _make_opt(model):
    opt = torch.optim.AdamW(model.parameters(), lr=float(TRAIN_TAG['lr']), weight_decay=float(TRAIN_TAG['weight_decay']))
    sch = baseline_tool.get_scheduler(opt, int(TRAIN_TAG['warmup_epochs']), int(TRAIN_TAG['epochs']))
    return opt, sch


def _train_standard(name: str, model, train_loader, val_loader, *, feat_key: str, mask_key: str):
    opt, sch = _make_opt(model)
    best_auc, best_state, no_imp = 0.0, None, 0
    for ep in range(1, int(TRAIN_TAG['epochs']) + 1):
        loss, _ = baseline_tool.train_standard(model, train_loader, opt, device, feat_key, mask_key)
        sch.step()
        val_auc, _, _ = baseline_tool.evaluate(model, val_loader, device, feat_key, mask_key)
        if val_auc > best_auc + 1e-4:
            best_auc = float(val_auc)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if ep % 2 == 0 or ep == 1:
            print(f"[{name}] ep={ep:03d} train_loss={loss:.5f} val_auc={val_auc:.5f} best={best_auc:.5f} no_imp={no_imp}")
        if no_imp >= int(TRAIN_TAG['patience']):
            print(f"[{name}] Early stopping")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def _train_kd(name: str, student, teacher, train_loader, val_loader):
    opt, sch = _make_opt(student)
    best_auc, best_state, no_imp = 0.0, None, 0
    for ep in range(1, int(TRAIN_TAG['epochs']) + 1):
        loss, _ = baseline_tool.train_kd(student, teacher, train_loader, opt, device, KD_CFG)
        sch.step()
        val_auc, _, _ = baseline_tool.evaluate(student, val_loader, device, 'hlt', 'mask_hlt')
        if val_auc > best_auc + 1e-4:
            best_auc = float(val_auc)
            best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if ep % 2 == 0 or ep == 1:
            print(f"[{name}] ep={ep:03d} train_loss={loss:.5f} val_auc={val_auc:.5f} best={best_auc:.5f} no_imp={no_imp}")
        if no_imp >= int(TRAIN_TAG['patience']):
            print(f"[{name}] Early stopping")
            break
    if best_state is not None:
        student.load_state_dict(best_state)
    return student


# -----------------------------
# 3) Train / load models (with checkpoints)
# -----------------------------
print('\n=== Downstream training ===')

SHARED_TAG_CKPT_DIR = SHARED_BASELINE_CKPT_DIR
UNSMEAR_TAG_CKPT_DIR = os.path.join(CKPT_DIR, 'downstream_tagger')
tool.ensure_dir(UNSMEAR_TAG_CKPT_DIR)

# Save metadata to avoid loading checkpoints from incompatible configurations.
_SPLIT_DIGEST = hashlib.sha1(np.asarray(train_idx, dtype=np.int64).tobytes()).hexdigest()[:12]
_BASELINE_META = {
    'feature_kind': str(DOWN_KIND),
    'tagger_cfg': dict(TAGGER_CFG),
    'kd_cfg': dict(KD_CFG),
    'split_digest': str(_SPLIT_DIGEST),
    'data_path': str(CONFIG['data_path']),
    'n_jets': int(CONFIG['n_jets']),
    'max_particles': int(CONFIG['max_particles']),
    'hlt_effects': dict(CONFIG['hlt_effects']),
}
_UNSMEAR_META = {
    'run_name': str(RUN_NAME),
    'downstream_feature_kind': str(DOWN_KIND),
    'downstream_post3_source': str(POST3_SOURCE),
    'unsmear_feature_kind': str(CONFIG.get('feature_kind')),
    'tagger_cfg': dict(TAGGER_CFG),
    'kd_cfg': dict(KD_CFG),
    'split_digest': str(_SPLIT_DIGEST),
    'mlp_cfg': dict(CONFIG['mlp']),
    'transformer_cfg': dict(CONFIG['transformer']),
    'unet_cfg': dict(CONFIG['unet']),
    'fm_cfg': dict(CONFIG['fm']),
    'hlt_effects': dict(CONFIG['hlt_effects']),
}


def _try_load_shared(model, ckpt_path: str) -> bool:
    if not bool(CONFIG.get('load_shared_baselines', True)):
        return False
    if not os.path.isfile(ckpt_path):
        return False
    obj = torch.load(ckpt_path, map_location='cpu')
    if isinstance(obj, dict) and 'meta' in obj:
        meta = obj.get('meta') or {}
        for k in ['feature_kind', 'tagger_cfg', 'kd_cfg', 'split_digest', 'data_path', 'n_jets', 'max_particles', 'hlt_effects']:
            if str(meta.get(k)) != str(_BASELINE_META.get(k)):
                print('Skip shared checkpoint (meta mismatch):', ckpt_path)
                print('  want', k, '=', _BASELINE_META.get(k))
                print('  got ', k, '=', meta.get(k))
                return False
    state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
    model.load_state_dict(state)
    print('Loaded shared checkpoint:', ckpt_path)
    return True


def _save_shared(model, ckpt_path: str, *, extra: dict | None = None):
    payload = {
        'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'meta': dict(_BASELINE_META),
    }
    if extra is not None:
        payload['extra'] = dict(extra)
    torch.save(payload, ckpt_path)
    print('Saved shared checkpoint:', ckpt_path)


def _try_load_unsmear_bundle(model, ckpt_path: str) -> bool:
    if not bool(CONFIG.get('load_unsmear_bundle', False)):
        return False
    if not os.path.isfile(ckpt_path):
        return False
    obj = torch.load(ckpt_path, map_location='cpu')
    if isinstance(obj, dict) and 'meta' in obj:
        meta = obj.get('meta') or {}
        for k in ['run_name', 'downstream_feature_kind', 'downstream_post3_source', 'unsmear_feature_kind', 'tagger_cfg', 'kd_cfg', 'split_digest', 'mlp_cfg', 'transformer_cfg', 'unet_cfg', 'fm_cfg', 'hlt_effects']:
            if str(meta.get(k)) != str(_UNSMEAR_META.get(k)):
                print('Skip unsmear checkpoint (meta mismatch):', ckpt_path)
                print('  want', k, '=', _UNSMEAR_META.get(k))
                print('  got ', k, '=', meta.get(k))
                return False
    state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
    model.load_state_dict(state)
    print('Loaded unsmear checkpoint:', ckpt_path)
    return True


def _save_unsmear_bundle(model, ckpt_path: str, *, extra: dict | None = None):
    payload = {
        'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'meta': dict(_UNSMEAR_META),
    }
    if extra is not None:
        payload['extra'] = dict(extra)
    torch.save(payload, ckpt_path)
    print('Saved unsmear checkpoint:', ckpt_path)


def _train_or_load_standard(name: str, ckpt_name: str, train_loader, val_loader, *, mode: str):
    model = ParticleTransformerKD(**TAGGER_CFG).to(device)
    ckpt_dir = SHARED_TAG_CKPT_DIR if mode == 'shared' else UNSMEAR_TAG_CKPT_DIR
    load_fn = _try_load_shared if mode == 'shared' else _try_load_unsmear_bundle
    save_fn = _save_shared if mode == 'shared' else _save_unsmear_bundle
    ckpt = os.path.join(ckpt_dir, ckpt_name)
    if not load_fn(model, ckpt):
        feat_key = 'off' if 'Teacher' in name else 'hlt'
        mask_key = 'mask_off' if 'Teacher' in name else 'mask_hlt'
        model = _train_standard(name, model, train_loader, val_loader, feat_key=feat_key, mask_key=mask_key)
        save_fn(model, ckpt)
    return model


def _train_or_load_kd(name: str, ckpt_name: str, teacher, train_loader, val_loader, *, mode: str):
    model = ParticleTransformerKD(**TAGGER_CFG).to(device)
    ckpt_dir = SHARED_TAG_CKPT_DIR if mode == 'shared' else UNSMEAR_TAG_CKPT_DIR
    load_fn = _try_load_shared if mode == 'shared' else _try_load_unsmear_bundle
    save_fn = _save_shared if mode == 'shared' else _save_unsmear_bundle
    ckpt = os.path.join(ckpt_dir, ckpt_name)
    if not load_fn(model, ckpt):
        model = _train_kd(name, model, teacher, train_loader, val_loader)
        save_fn(model, ckpt)
    return model


teacher = _train_or_load_standard('Teacher(OFF_FULL)', 'teacher_offline.pt', train_loader_hlt, val_loader_hlt, mode='shared')
student_hlt = _train_or_load_standard('Student(HLT)', 'student_hlt.pt', train_loader_hlt, val_loader_hlt, mode='shared')
student_mlp = _train_or_load_standard('Student(MLP+HLT)', f'tagger_student_mlp_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', train_loader_mlp, val_loader_mlp, mode='unsmear')
student_transformer = _train_or_load_standard('Student(Transformer+HLT)', f'tagger_student_transformer_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', train_loader_transformer, val_loader_transformer, mode='unsmear')
student_unet = _train_or_load_standard('Student(UNET+HLT)', f'tagger_student_unet_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', train_loader_unet, val_loader_unet, mode='unsmear')
student_fm = _train_or_load_standard('Student(FM+HLT)', f'tagger_student_fm_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', train_loader_fm, val_loader_fm, mode='unsmear')

student_hlt_kd = _train_or_load_kd('Student(HLT)+KD', 'student_hlt_kd.pt', teacher, train_loader_hlt, val_loader_hlt, mode='shared')
student_mlp_kd = _train_or_load_kd('Student(MLP+HLT)+KD', f'tagger_student_mlp_kd_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', teacher, train_loader_mlp, val_loader_mlp, mode='unsmear')
student_transformer_kd = _train_or_load_kd('Student(Transformer+HLT)+KD', f'tagger_student_transformer_kd_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', teacher, train_loader_transformer, val_loader_transformer, mode='unsmear')
student_unet_kd = _train_or_load_kd('Student(UNET+HLT)+KD', f'tagger_student_unet_kd_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', teacher, train_loader_unet, val_loader_unet, mode='unsmear')
student_fm_kd = _train_or_load_kd('Student(FM+HLT)+KD', f'tagger_student_fm_kd_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt', teacher, train_loader_fm, val_loader_fm, mode='unsmear')

# -----------------------------
# 4) Evaluate + ROC
# -----------------------------
print('\n=== Test evaluation ===')

auc_t, p_t, y_t = baseline_tool.evaluate(teacher, test_loader_hlt, device, 'off', 'mask_off')
auc_hlt, p_hlt, _ = baseline_tool.evaluate(student_hlt, test_loader_hlt, device, 'hlt', 'mask_hlt')
auc_mlp, p_mlp, _ = baseline_tool.evaluate(student_mlp, test_loader_mlp, device, 'hlt', 'mask_hlt')
auc_transformer, p_transformer, _ = baseline_tool.evaluate(student_transformer, test_loader_transformer, device, 'hlt', 'mask_hlt')
auc_unet, p_unet, _ = baseline_tool.evaluate(student_unet, test_loader_unet, device, 'hlt', 'mask_hlt')
auc_fm, p_fm, _ = baseline_tool.evaluate(student_fm, test_loader_fm, device, 'hlt', 'mask_hlt')
auc_hlt_kd, p_hlt_kd, _ = baseline_tool.evaluate(student_hlt_kd, test_loader_hlt, device, 'hlt', 'mask_hlt')
auc_mlp_kd, p_mlp_kd, _ = baseline_tool.evaluate(student_mlp_kd, test_loader_mlp, device, 'hlt', 'mask_hlt')
auc_transformer_kd, p_transformer_kd, _ = baseline_tool.evaluate(student_transformer_kd, test_loader_transformer, device, 'hlt', 'mask_hlt')
auc_unet_kd, p_unet_kd, _ = baseline_tool.evaluate(student_unet_kd, test_loader_unet, device, 'hlt', 'mask_hlt')
auc_fm_kd, p_fm_kd, _ = baseline_tool.evaluate(student_fm_kd, test_loader_fm, device, 'hlt', 'mask_hlt')

print(f'Teacher(OFF_FULL) AUC={auc_t:.5f}')
print(f'Student(HLT) AUC={auc_hlt:.5f}')
print(f'Student(MLP+HLT) AUC={auc_mlp:.5f}')
print(f'Student(Transformer+HLT) AUC={auc_transformer:.5f}')
print(f'Student(UNET+HLT) AUC={auc_unet:.5f}')
print(f'Student(FM+HLT) AUC={auc_fm:.5f}')
print(f'Student(HLT)+KD AUC={auc_hlt_kd:.5f}')
print(f'Student(MLP+HLT)+KD AUC={auc_mlp_kd:.5f}')
print(f'Student(Transformer+HLT)+KD AUC={auc_transformer_kd:.5f}')
print(f'Student(UNET+HLT)+KD AUC={auc_unet_kd:.5f}')
print(f'Student(FM+HLT)+KD AUC={auc_fm_kd:.5f}')

curves = {
    'Teacher(OFF_FULL)': (p_t, y_t, auc_t),
    'Student(HLT)': (p_hlt, y_t, auc_hlt),
    'Student(MLP+HLT)': (p_mlp, y_t, auc_mlp),
    'Student(Transformer+HLT)': (p_transformer, y_t, auc_transformer),
    'Student(UNET+HLT)': (p_unet, y_t, auc_unet),
    'Student(FM+HLT)': (p_fm, y_t, auc_fm),
    'Student(HLT)+KD': (p_hlt_kd, y_t, auc_hlt_kd),
    'Student(MLP+HLT)+KD': (p_mlp_kd, y_t, auc_mlp_kd),
    'Student(Transformer+HLT)+KD': (p_transformer_kd, y_t, auc_transformer_kd),
    'Student(UNET+HLT)+KD': (p_unet_kd, y_t, auc_unet_kd),
    'Student(FM+HLT)+KD': (p_fm_kd, y_t, auc_fm_kd),
}

plt.figure(figsize=(8.6, 6.2))
for name, (p, y, aucv) in curves.items():
    fpr, tpr, _auc = baseline_tool.compute_roc(y, p)
    plt.semilogy(tpr, np.clip(fpr, 1e-6, 1.0), lw=2, label=f'{name} (AUC={aucv:.4f})')

plt.xlabel('TPR')
plt.ylabel('FPR (log)')
plt.title('Downstream ROC (TPR vs log FPR)')
plt.grid(True, which='both', alpha=0.25)
plt.legend(loc='upper right', fontsize=8)
plt.xlim(0.0, 1.0)
plt.ylim(1e-4, 1.0)

out = os.path.join(FIG_DIR, 'downstream_roc_logfpr.png')
plt.tight_layout()
plt.savefig(out, dpi=170, bbox_inches='tight')
print('Saved figure:', out)
plt.show()
